# From tweets to figures — the whole Chapter 15 pipeline in one notebook

This notebook runs every step of the analysis in Chapter 15, *Computational
Methods for Studying Online Polarization: An Arabic Social Media Case Study*,
from a folder of tweets to the printed figures. No number below is read from a
saved results table; everything is computed while you watch.

Set `MODE` in the next cell.

| `MODE` | What it does |
|---|---|
| `"chapter"` | Reproduces the chapter: loads the trained model and the published category mapping, then recomputes the assignments, shares, retention table and both figures from the tweets. |
| `"own"` | Runs the same pipeline on **your** tweets: trains a fresh model with a fixed seed, prints the topics, and leaves the grouping to you. |

**Why two modes.** The chapter's model was trained without a random seed, so
retraining cannot reproduce it — a second run gives different topics in a
different order. `"chapter"` therefore loads the saved model. `"own"` sets
`random_state`, so your run *is* repeatable. This is the one place the notebook
departs from the original code, and §10 says why.

**The one step a computer cannot do.** Step 12 groups fifty statistical topics
into interpretable categories. That is a reading judgment. In `"chapter"` mode
the notebook uses the published mapping; in `"own"` mode it prints the topics and
waits for yours. Everything after step 12 is arithmetic.

**Taken as given.** The four community labels, which come from a previously
published study that built them from the retweet graph. If your tweets carry no
labels, the corpus is treated as one group and the pipeline still runs.

**Time.** Farasa is the slow step — the original code notes about 2.6 seconds per
tweet, so the full corpus is hours, not minutes. Set `SAMPLE_N` to a few thousand
to walk the chain first. Every stage caches, so stopping and restarting is safe.

---
## 1. Settings

This is the only cell you need to edit.

In [ ]:
# ── what to run ───────────────────────────────────────────────────────────────
MODE     = "chapter"    # "chapter" = reproduce Chapter 15 | "own" = your own tweets
SAMPLE_N = None         # None = all tweets; e.g. 5000 = a quick walk through the chain
SEED     = 400          # used for sampling, and for training in "own" mode

# ── where the tweets are ──────────────────────────────────────────────────────
# A .json dict of tweets (the chapter's format), or .jsonl / .csv with one row
# per tweet. Required field: the tweet text. Optional but needed for the figures:
# a timestamp, and a community label.
# DATA_ROOT: the defaults below assume this notebook sits in
# chapter15-practices-public, inside the author's lda_data folder, so they point
# one level up at the corpus, the model and the Arabic resources. None of those
# files are in the public repository — they are too large — so a reader who
# cloned the repo will see step 2 report them missing. That is expected: set
# TWEETS_FILE to your own tweets and MODE to "own".
DATA_ROOT = ".."

TWEETS_FILE = f"{DATA_ROOT}/data/tweets_with_labels.json"

TEXT_FIELD      = "full_text"         # the tweet text
DATE_FIELD      = "tweet_date_time"   # timestamp; None = skip the monthly figures
COMMUNITY_FIELD = "retweeters"        # see step 13; None = treat the corpus as one group

# ── supporting files ──────────────────────────────────────────────────────────
LANGID_MODEL = f"{DATA_ROOT}/assets/lid.176.bin"        # fastText language ID
FARASA_JAR   = f"{DATA_ROOT}/FarasaSegmenterJar.jar"    # only used by the Java fallback
STOPWORD_FILES = [f"{DATA_ROOT}/stopwords/StopDialect.txt",
                  f"{DATA_ROOT}/stopwords/StopDialect_stemmed.txt"]
KEEP_LANG    = "ar"      # keep only this language; None = keep everything
LANG_FALLBACK = True     # with no fastText, fall back to an Arabic-script test
ARABIC_MIN    = 0.5      # the fallback's threshold: Arabic share of the letters
MIN_TOKENS   = 3         # drop tweets shorter than this many whitespace tokens

# ── the model ─────────────────────────────────────────────────────────────────
NUM_TOPICS   = 50
PASSES       = 20
WORKERS      = 8
NO_BELOW     = 5         # drop a word seen in fewer than this many tweets
NO_ABOVE     = 0.02      # drop a word seen in more than this share of tweets

# "chapter" mode loads these instead of training.
# There is no separate dictionary setting on purpose: the vocabulary is taken
# from inside the model file. See step 9.
SAVED_MODEL      = f"{DATA_ROOT}/lda_gensim_model_50.obj"
PUBLISHED_PKG    = "."                 # the folder holding topic_groups.py

# ── output ────────────────────────────────────────────────────────────────────
CACHE_DIR  = "./az_cache"     # intermediate results, safe to delete
OUTPUT_DIR = "./az_output"    # the figures and CSVs land here

assert MODE in ("chapter", "own"), 'MODE must be "chapter" or "own"'
print(f"MODE = {MODE}   SAMPLE_N = {SAMPLE_N}   SEED = {SEED}")

---
## 2. What is installed

The pipeline needs four outside pieces: **gensim** (topic model), **fastText**
(language ID), **Farasa** (Arabic stemming, needs Java) and **matplotlib**
(figures). The cell below reports which ones are present and tells you exactly
what will happen to any step whose tool is missing. Nothing is installed for you.

In [ ]:
import importlib, os, shutil, subprocess, sys, textwrap

def _probe(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

HAVE = {m: _probe(m) for m in ("gensim", "pandas", "numpy", "matplotlib",
                               "fasttext", "farasa", "nltk")}
HAVE["java"] = shutil.which("java") is not None

FILES = {"tweets": TWEETS_FILE, "langid model": LANGID_MODEL,
         "Farasa jar": FARASA_JAR, "saved LDA model": SAVED_MODEL,
         "published package": PUBLISHED_PKG}
FILES.update({f"stopwords[{i}]": p for i, p in enumerate(STOPWORD_FILES)})
PRESENT = {k: os.path.exists(v) for k, v in FILES.items()}

print("packages")
for k, v in HAVE.items():
    print(f"   {'yes' if v else 'NO '}  {k}")
print("\nfiles")
for k, v in FILES.items():
    print(f"   {'yes' if PRESENT[k] else 'NO '}  {k:18} {v}")

notes = []
if not HAVE["gensim"]:
    notes.append("gensim is missing. Steps 9-17 cannot run. "
                 "Everything up to the bag of words still works.")
if not HAVE["pandas"]:
    notes.append("pandas is missing. Nothing below step 3 can run.")
if not (HAVE["fasttext"] and PRESENT["langid model"]):
    notes.append("No fastText language ID. Step 4 falls back to an Arabic-script "
                 "test, which keeps a different set of tweets than the chapter "
                 "did, so the corpus below will not be the published one.")
if not (HAVE["java"] and PRESENT["Farasa jar"]):
    if HAVE["farasa"]:
        notes.append("The Farasa jar is not usable, so step 7 will try the "
                     "farasa Python package. The published model was trained "
                     "through the jar.")
    else:
        notes.append("NO STEMMER. Step 7 will hand the model unstemmed tokens. "
                     "The model's vocabulary is made of stems, so most words "
                     "will not match it and the topic assignments will be "
                     "close to meaningless. This is not a small loss of "
                     "accuracy; it breaks the analysis.")
if not HAVE["nltk"]:
    notes.append("nltk is missing, so step 7 tokenizes with a regular "
                 "expression instead of TweetTokenizer. Close, not identical.")
if not HAVE["matplotlib"]:
    notes.append("matplotlib is missing. Steps 15-16 print their tables but "
                 "draw no figures.")
if MODE == "chapter":
    for k in ("saved LDA model", "published package"):
        if not PRESENT[k]:
            notes.append(f'MODE is "chapter" but {k} is missing ({FILES[k]}). '
                         f'Either put it in place or set MODE = "own".')
    blockers = []
    if not (HAVE["fasttext"] and PRESENT["langid model"]):
        blockers.append("fastText language ID")
    if not (HAVE["java"] and PRESENT["Farasa jar"]):
        blockers.append("the Farasa jar")
    if blockers:
        notes.append('MODE is "chapter", which claims to reproduce the '
                     'published figures, but ' + " and ".join(blockers) +
                     " cannot be used here. The preprocessing will not be the "
                     "preprocessing behind the chapter, so step 18 will report "
                     "a disagreement. Read that number rather than the figures.")

print()
if notes:
    print("read these before you continue")
    for n in notes:
        print(textwrap.fill(n, 78, initial_indent="  - ", subsequent_indent="    "))
else:
    print("everything the pipeline needs is present.")

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import json, pickle, re, time
from datetime import datetime

def cache(name):
    return os.path.join(CACHE_DIR, f"{name}.pkl")

def cached(name, build, label=None):
    """Run build() once, then reuse. Keys carry what the result depends on —
    mode, tweet count, vocabulary size — so changing any of them rebuilds
    instead of reusing a stale result. Delete a file in az_cache to force it."""
    path = cache(name)
    if os.path.exists(path):
        with open(path, "rb") as fh:
            out = pickle.load(fh)
        print(f"  [cache] {label or name}")
        return out
    t0 = time.time()
    out = build()
    with open(path, "wb") as fh:
        pickle.dump(out, fh)
    print(f"  [built] {label or name} in {time.time()-t0:.1f}s")
    return out

print(f"cache  -> {os.path.abspath(CACHE_DIR)}")
print(f"output -> {os.path.abspath(OUTPUT_DIR)}")

---
## 3. Load the tweets

The chapter's file is a single JSON object keyed by tweet id. The loader also
accepts JSON Lines and CSV, so you can point it at your own export. It reports
the fields it found rather than assuming a layout.

In [ ]:
import pandas as pd

def load_tweets(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Set TWEETS_FILE in step 1 to your tweets file.")
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in (".jsonl", ".ndjson"):
        return pd.read_json(path, lines=True)
    if ext in (".pkl", ".pickle"):
        return pd.read_pickle(path)
    with open(path, "r", encoding="utf-8") as fh:
        head = fh.read(1); fh.seek(0)
        if head == "[":
            return pd.DataFrame(json.load(fh))
        obj = json.loads(fh.readline())          # the chapter's one-line format
    return pd.DataFrame.from_dict(obj, orient="index")

raw = load_tweets(TWEETS_FILE)
print(f"{len(raw):,} tweets loaded from {TWEETS_FILE}")
print(f"\ncolumns: {', '.join(map(str, raw.columns))}\n")

for field, name in ((TEXT_FIELD, "TEXT_FIELD"), (DATE_FIELD, "DATE_FIELD"),
                    (COMMUNITY_FIELD, "COMMUNITY_FIELD")):
    if field is None:
        print(f"{name:16} not set -> the steps that need it will be skipped")
    elif field in raw.columns:
        print(f"{name:16} '{field}' found")
    else:
        print(f"{name:16} '{field}' NOT in this file  <-- fix step 1")

if TEXT_FIELD not in raw.columns:
    raise KeyError(f"'{TEXT_FIELD}' is not a column. Set TEXT_FIELD in step 1.")
raw.head(3)

In [ ]:
# The chapter's raw text carries a leading id and tab, and hard line breaks.
# Both are stripped here exactly as the original preprocessing did.
def clean_text(s):
    s = "" if s is None else str(s)
    if "\t" in s:
        s = " ".join(s.split("\t")[1:])      # drop the id prefix
    return re.sub(r"[\n ]+", " ", s).strip()

df = raw.copy()
df["text"] = df[TEXT_FIELD].map(clean_text)

before = len(df)
df = df[df["text"].str.len() > 0]
print(f"{before - len(df):,} tweets had no text after cleaning; {len(df):,} remain")

if DATE_FIELD and DATE_FIELD in df.columns:
    df["when"] = pd.to_datetime(df[DATE_FIELD], errors="coerce")
    bad = df["when"].isna().sum()
    df = df[df["when"].notna()]
    print(f"{bad:,} tweets had an unreadable timestamp; {len(df):,} remain")
    print(f"the corpus runs {df['when'].min():%Y-%m-%d} .. {df['when'].max():%Y-%m-%d}")

print(f"\nexample: {df['text'].iloc[0][:160]}")

---
## 4. Keep one language

fastText's `lid.176.bin` labels each tweet's language and anything that is not
`KEEP_LANG` is dropped. If fastText or the model file is absent this step is
skipped and says so — it does not quietly pretend to have run.

In [ ]:
ARABIC = re.compile(r"[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]")
LATIN  = re.compile(r"[A-Za-z]")

def arabic_share(s):
    """Arabic letters as a share of all letters. Links and digits ignored."""
    s = re.sub(r"https?://\S+", " ", s)
    a, l = len(ARABIC.findall(s)), len(LATIN.findall(s))
    return a / (a + l) if (a + l) else 0.0


def language_filter(frame):
    if KEEP_LANG is None:
        print("KEEP_LANG is None -> keeping every language"); return frame

    if not (HAVE["fasttext"] and PRESENT["langid model"]):
        if KEEP_LANG != "ar" or not LANG_FALLBACK:
            print("SKIPPED: no fastText language ID. Every tweet kept.")
            print("         On a mixed-language corpus this changes the topics.")
            return frame
        print("fastText is not installed. Falling back to a script test:")
        print(f"a tweet is kept when Arabic letters are at least "
              f"{ARABIC_MIN*100:.0f}% of its letters.")
        print("This is NOT what the chapter used. It will keep or drop a")
        print("different set of tweets, so the corpus — and every number")
        print("below — will differ from the published ones.")
        share = frame["text"].map(arabic_share)
        out = frame[share >= ARABIC_MIN]
        print(f"\n{len(frame):,} tweets, of which {len(out):,} are mostly "
              f"Arabic ({len(frame)-len(out):,} dropped)")
        return out

    import fasttext
    lid = fasttext.load_model(LANGID_MODEL)
    # predict() takes a LIST: faster, and it avoids the single-string branch,
    # which ends in np.array(probs, copy=False) and fails under NumPy 2.
    # Newlines raise, so they are flattened; the call is chunked for memory.
    texts = frame["text"].str.replace(r"[\r\n]+", " ", regex=True).tolist()
    langs, step, t0 = [], 200_000, time.time()
    for lo in range(0, len(texts), step):
        labels, _ = lid.predict(texts[lo:lo + step])
        langs += [l[0].replace("__label__", "") if l else "" for l in labels]
        print(f"   {min(lo+step, len(texts)):,}/{len(texts):,}  "
              f"{time.time()-t0:.0f}s")
    keep = [l == KEEP_LANG for l in langs]
    out = frame[keep]
    print(f"{len(frame):,} tweets, of which {len(out):,} are '{KEEP_LANG}' "
          f"({len(frame)-len(out):,} dropped)")
    top = pd.Series(langs).value_counts().head(6)
    print("\nmost common languages found:")
    print(top.to_string())
    return out

# The key names the method, so switching from "no filter" to the script test —
# or later to fastText — rebuilds instead of silently reusing the old corpus.
LANG_TAG = ("fasttext" if (HAVE["fasttext"] and PRESENT["langid model"])
            else "arabic" if (KEEP_LANG == "ar" and LANG_FALLBACK)
            else "none")
df = cached(f"step4_language_{LANG_TAG}", lambda: language_filter(df),
            f"language filter ({LANG_TAG})")
print(f"\n{len(df):,} tweets after step 4")

---
## 5. Drop the very short tweets

A tweet of one or two words gives a topic model almost nothing to work with. The
original code dropped anything under **three** whitespace tokens, and that is the
threshold reported in §15.3 of the chapter.

In [ ]:
lengths = df["text"].str.split(" ").str.len()
before = len(df)
df = df[lengths >= MIN_TOKENS]
print(f"{before:,} tweets, of which {len(df):,} have at least {MIN_TOKENS} "
      f"tokens ({before-len(df):,} dropped)")
print("\nlength in whitespace tokens, before the cut:")
print(lengths.describe().round(1).to_string())

In [ ]:
# Optional: work on a sample so the whole chain finishes in minutes.
if SAMPLE_N and SAMPLE_N < len(df):
    df = df.sample(n=SAMPLE_N, random_state=SEED)
    print(f"SAMPLE_N is set: working on {len(df):,} tweets drawn with seed {SEED}.")
    print("Numbers below describe the sample, not the whole corpus.")
else:
    print(f"working on all {len(df):,} tweets")
df = df.reset_index(drop=True)

---
## 6. The stop-word list

Two lists, surface forms and stemmed forms, merged and de-duplicated, exactly as
the original code read them.

In [ ]:
def load_stopwords(paths):
    words = []
    for p in paths:
        if not os.path.exists(p):
            print(f"   missing: {p}")
            continue
        with open(p, "r", encoding="utf-8") as fh:
            got = [w.strip() for w in fh if w.strip()]
        words += got
        print(f"   {len(got):,} from {p}")
    return set(words)

STOPWORDS = load_stopwords(STOPWORD_FILES)
print(f"\n{len(STOPWORDS):,} distinct stop words")
if not STOPWORDS:
    print("WARNING: the list is empty, so no stop words will be removed.")

---
## 7. Tokenize and stem

Arabic marks person, number and definiteness on the word itself, so without
stemming one idea is spread over a dozen surface forms and the model sees a dozen
weak words instead of one strong one. This is not a refinement: the model's
vocabulary is a vocabulary of *stems*, and unstemmed tweets barely match it.

The chapter's pipeline, which is what the saved model was trained on, does four
things: tokenize and set hashtags and mentions aside (a hashtag is a label, not a
sentence); drop stop words and single characters from the rest; hand the
remainder to **Farasa** through `FarasaSegmenterJar.jar`, which needs Java; then
put the hashtags and mentions back beside the stems.

The cell below does the same, preferring the jar because that is what produced
the published model. Failing that it tries the `farasa` Python package, and
failing that it falls back to unstemmed tokens — which lets you watch the
pipeline run but produces topics unrelated to the chapter's.

In [ ]:
try:
    from nltk.tokenize import TweetTokenizer
    _tok = TweetTokenizer().tokenize
    print("tokenizer: nltk TweetTokenizer (what the chapter used)")
except Exception:
    _TOKEN = re.compile(r"[@#]?[\w\u0600-\u06FF_]+|\S")
    _tok = _TOKEN.findall
    print("tokenizer: nltk is missing, using a regular expression instead.")
    print("It keeps @ and # attached, which is the part that matters here,")
    print("but it will not split every edge case the way nltk does.")


def split_tweet(text):
    """-> (words to stem, mentions, hashtags) — the chapter's cells 14-15."""
    words, mentions, hashtags = [], [], []
    for tok in _tok(text):
        tok = tok.strip()
        if len(tok) <= 1:
            continue
        if tok.startswith("#"):
            hashtags.append(tok)
        elif tok.startswith("@"):
            mentions.append(tok)
        elif tok not in STOPWORDS:
            words.append(tok)
    return " ".join(words), mentions, hashtags


parts = [split_tweet(t) for t in df["text"]]
df["words"]    = [p[0] for p in parts]
df["mentions"] = [p[1] for p in parts]
df["hashtags"] = [p[2] for p in parts]
del parts
print(f"\nsplit {len(df):,} tweets")
print(f"example words   : {df['words'].iloc[0][:120]}")
print(f"example hashtags: {df['hashtags'].iloc[0]}")

In [ ]:
# Which stemmer is available, in the order of preference.
STEM_MODE = None
if HAVE["java"] and PRESENT["Farasa jar"]:
    STEM_MODE = "jar"
    print("stemmer: the Farasa jar, run through Java.")
    print("This is the one the published model was trained on.")
elif HAVE["farasa"]:
    try:
        from farasa import stemmer as _stm
        _PY_STEMMER = _stm.FarasaStemmer()
        STEM_MODE = "python"
        print("stemmer: the farasa Python package.")
        print("The published model used the jar; results may differ slightly.")
    except Exception as exp:
        print(f"the farasa package would not start: {exp}")

if STEM_MODE is None:
    print("NO STEMMER AVAILABLE.")
    print("Tokens go to the model unstemmed. The model's vocabulary is a")
    print("vocabulary of stems, so most words will not match it and the topic")
    print("assignments will be close to meaningless. Install Java and put")
    print("FarasaSegmenterJar.jar beside the notebook before trusting anything")
    print("below.")
if MODE == "chapter" and STEM_MODE != "jar":
    print("\nMODE is \"chapter\" but the jar is not being used, so this run")
    print("cannot reproduce the published numbers whatever else it does.")

In [ ]:
def stem_with_jar():
    """Write one 'rowid,text' line per tweet, run the jar, read the stems back."""
    src = os.path.join(CACHE_DIR, "farasa_in.csv")
    dst = os.path.join(CACHE_DIR, "farasa_out.txt")
    jar = os.path.abspath(FARASA_JAR)

    with open(src, "w", encoding="utf-8") as fh:
        for i, w in enumerate(df["words"]):
            fh.write(f"{i},{w}\n")
    print(f"   wrote {len(df):,} lines to {src}")

    if not os.path.exists(dst):
        cmd = ["java", "-jar", jar, "-l", "true", "-i", src, "-o", dst]
        print("   " + " ".join(cmd))
        print("   this is the long step; Farasa prints its own progress.")
        rc = subprocess.call(cmd)
        if rc != 0 or not os.path.exists(dst):
            raise SystemExit(
                f"Farasa exited with code {rc} and wrote no output. Check that "
                f"java runs ('java -version') and that {jar} is the segmenter "
                f"jar. Delete {dst} before trying again.")
    else:
        print(f"   reusing {dst}; delete it to run Farasa again")

    # Farasa keeps the leading row id as the first token on the line.
    stems = {}
    with open(dst, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.strip().replace(",", " ")
            if not line:
                continue
            head, _, rest = line.partition(" ")
            try:
                stems[int(head.strip())] = rest
            except ValueError:
                continue
    print(f"   read {len(stems):,} stemmed lines back")
    if len(stems) < len(df) * 0.9:
        print(f"   WARNING: only {len(stems)/len(df)*100:.1f}% of tweets came "
              f"back. The rest will carry hashtags and mentions only.")
    return [stems.get(i, "") for i in range(len(df))]


def stem_all():
    if STEM_MODE == "jar":
        stemmed = stem_with_jar()
    elif STEM_MODE == "python":
        stemmed, t0 = [], time.time()
        for i, w in enumerate(df["words"], 1):
            try:
                stemmed.append(_PY_STEMMER.stem(w) if w else "")
            except Exception:
                stemmed.append(w)
            if i % 20000 == 0:
                print(f"   {i:,}/{len(df):,}  {i/max(time.time()-t0,1e-9):,.0f}/s")
    else:
        stemmed = list(df["words"])
    # put the hashtags and mentions back: the chapter's cell 30
    return [(st + " " + " ".join(m) + " " + " ".join(h)).split()
            for st, m, h in zip(stemmed, df["mentions"], df["hashtags"])]


df["tokens"] = cached(f"step7_tokens_{STEM_MODE}_{len(df)}", stem_all, "stemming")
print(f"\n{len(df):,} tweets tokenized")
print(f"example: {df['tokens'].iloc[0]}")

---
## 8. Remove numbers, single letters and any stop word the stemmer produced

Stemming can turn a content word into something that belongs on the stop-word
list, so the list is applied a second time, after stemming, together with a cut
for digits and single characters.

In [ ]:
def keep_token(t):
    return len(t) > 1 and not t.isdigit() and t not in STOPWORDS

df["processed_docs"] = [[t for t in toks if keep_token(t)] for toks in df["tokens"]]

before_empty = len(df)
df = df[df["processed_docs"].str.len() > 0].reset_index(drop=True)
print(f"{before_empty - len(df):,} tweets had nothing left and were dropped")
print(f"{len(df):,} tweets go to the model")

sizes = df["processed_docs"].str.len()
print(f"\ntokens per tweet: mean {sizes.mean():.1f}, median {sizes.median():.0f}, "
      f"max {sizes.max()}")
from collections import Counter
common = Counter(t for toks in df["processed_docs"] for t in toks).most_common(15)
print("\nmost frequent tokens in the corpus:")
for w, n in common:
    print(f"   {n:>9,}  {w}")

---
## 9. Build the vocabulary

Two filters do most of the work of making topics readable. `no_below=5` drops a
word seen in fewer than five tweets, removing typos and one-off coinages.
`no_above=0.02` drops a word seen in more than two per cent of tweets, removing
the words so common they would otherwise appear in every topic at once.

In `"chapter"` mode the vocabulary is taken **from inside the model**, not from a
dictionary file. A trained LDA model is a matrix of topics by words whose columns
are indexed against the exact vocabulary it was trained on, and a dictionary
saved in the same folder need not be that vocabulary — the author's
`data/dictionary_object` holds about 209,000 words against the model's 140,926.
Feeding the larger one's bags of words to the smaller model raises

    IndexError: index 209041 is out of bounds for axis 1 with size 140926

So the model is loaded here, a step earlier than the narrative suggests, purely
to get the vocabulary it actually knows.

In [ ]:
if not HAVE["gensim"]:
    raise SystemExit("gensim is required from here on. "
                     "Install it, restart the kernel and run again.")
import gensim

if MODE == "chapter":
    lda = gensim.models.LdaMulticore.load(SAVED_MODEL)
    dictionary = lda.id2word
    if not hasattr(dictionary, "doc2bow"):
        raise SystemExit(
            "The saved model carries a plain id->word map rather than a gensim "
            "Dictionary, so bags of words cannot be built from it. Retrain with "
            "id2word=<Dictionary>, or run in \"own\" mode.")
    print(f"the model's own vocabulary: {len(dictionary):,} words")
    print(f"(read out of {SAVED_MODEL}, not from any dictionary file — "
          f"see the note above)")
else:
    dictionary = gensim.corpora.Dictionary(df["processed_docs"].tolist())
    raw_vocab = len(dictionary)
    print(f"{raw_vocab:,} distinct words before filtering")
    dictionary.filter_extremes(no_below=NO_BELOW, no_above=NO_ABOVE)
    print(f"{len(dictionary):,} words after no_below={NO_BELOW}, "
          f"no_above={NO_ABOVE}")

    kept = len(dictionary) / raw_vocab if raw_vocab else 0
    if len(dictionary) < 200 or kept < 0.02:
        print(f"""
STOP AND READ. The vocabulary collapsed: {len(dictionary):,} words out of
{raw_vocab:,} ({kept*100:.1f}%). NO_ABOVE is a *share*, not a count: at 0.02 a
word is dropped once it appears in more than {int(0.02*len(df)):,} of your
{len(df):,} tweets. On 1.9 million that removes only the words common enough to
pollute every topic; on a small corpus it removes ordinary vocabulary.

Raise NO_ABOVE in step 1 (try 0.3, or 0.5 on a few thousand tweets), lower
NO_BELOW to 2, and run from step 9 again. Everything below is meaningless until
this line reports a few thousand words.""")

print("\nten words from the vocabulary:")
for i, (k, v) in enumerate(dictionary.iteritems()):
    print(f"   {k:>7}  {v}")
    if i >= 9:
        break

In [ ]:
# Bag of words: each tweet becomes a list of (word id, count).
def build_bows():
    return [dictionary.doc2bow(doc) for doc in df["processed_docs"].tolist()]

df["bow"] = cached(f"step9_bow_{MODE}_{len(df)}_{len(dictionary)}",
                   build_bows, "bag of words")

nonempty = sum(1 for b in df["bow"] if b)
share = nonempty / len(df) if len(df) else 0
print(f"\n{nonempty:,} of {len(df):,} tweets have at least one word in the vocabulary")
if nonempty < len(df):
    print(f"{len(df)-nonempty:,} are empty after filtering and carry no topic.")
if share < 0.5:
    print(f"""
STOP AND READ. Only {share*100:.1f}% of tweets survived the vocabulary filter,
so every number below would be computed from the remainder. Fix the filtering in
step 1 — see the previous cell — before going on.""")
print(f"\nthe first tweet as a bag of words:")
for wid, n in df["bow"].iloc[0][:8]:
    print(f"   {n} x {dictionary[wid]}")

---
## 10. The topic model

LDA treats every tweet as a mixture of topics and every topic as a distribution
over words, then looks for the fifty topics that best account for the whole
corpus at once. Nobody tells it what the topics are.

**On reproducibility.** The chapter's model was trained without `random_state`,
so retraining produces a different model: different topic order, different word
weights. `"chapter"` mode loads the saved model. `"own"` mode passes
`random_state=SEED`, so running it twice gives the same model both times — the
one deliberate change from the original code, and one worth keeping in your own
work.

In [ ]:
MODEL_PATH = os.path.join(CACHE_DIR, f"lda_{NUM_TOPICS}_seed{SEED}.obj")

if MODE == "chapter":
    # already loaded in step 9, which needed its vocabulary
    print(f"using {SAVED_MODEL}, loaded in step 9")
    print(f"{lda.num_topics} topics over a vocabulary of {len(lda.id2word):,}")
    print("\nThis model was trained without a random seed, so it cannot be")
    print("retrained identically. That is why it is loaded rather than rebuilt.")
elif os.path.exists(MODEL_PATH):
    lda = gensim.models.LdaMulticore.load(MODEL_PATH)
    print(f"reusing the model trained earlier: {MODEL_PATH}")
    print(f"delete that file to train again from scratch")
else:
    corpus = [b for b in df["bow"] if b]
    print(f"training {NUM_TOPICS} topics on {len(corpus):,} tweets, "
          f"{PASSES} passes, seed {SEED}.")
    print("this is the long step; progress goes to the kernel log.\n")
    t0 = time.time()
    lda = gensim.models.LdaMulticore(corpus, num_topics=NUM_TOPICS,
                                     id2word=dictionary, passes=PASSES,
                                     workers=WORKERS, random_state=SEED)
    lda.save(MODEL_PATH)
    print(f"trained in {(time.time()-t0)/60:.1f} minutes -> {MODEL_PATH}")

---
## 11. Read the topics

Here is what the model found. Each line is one topic and its strongest words.
Read them before you go on: this is the evidence you will use in step 12, and it
is also the honest place to notice a topic that is incoherent, duplicated, or
merely an artefact of the cleaning.

In [ ]:
TOPIC_WORDS = {t: [w for w, _ in lda.show_topic(t, 12)]
               for t in range(lda.num_topics)}

for t in range(lda.num_topics):
    print(f"topic {t:2d}  " + ", ".join(TOPIC_WORDS[t]))

with open(os.path.join(OUTPUT_DIR, f"topics_{lda.num_topics}.txt"),
          "w", encoding="utf-8") as fh:
    for t in range(lda.num_topics):
        fh.write(f"Topic {t}\n   " + ", ".join(TOPIC_WORDS[t]) + "\n")
print(f"\nwritten to {os.path.join(OUTPUT_DIR, f'topics_{lda.num_topics}.txt')}")

---
## 12. Group the topics into categories

**This step is yours.** Fifty topics are too many to compare across communities,
and several are facets of one subject. Grouping them is a reading judgment: you
look at the words printed in step 11 and decide that these four topics are all
football, those two are both illness and hardship, and this one is noise.

Two rules make the result defensible rather than decorative. **Write the mapping
down and publish it** — a grouping nobody can see is not a method; the chapter's
ships as `topic_groups.py`. And **keep a residual category** for topics that
belong nowhere, excluded from every denominator, so a share means "of the tweets
we could place" rather than "of everything".

`"chapter"` mode loads the published mapping. `"own"` mode gives you a template
to fill in and saves it beside your results.

In [ ]:
RESIDUAL = "Others"

if MODE == "chapter":
    sys.path.insert(0, os.path.abspath(PUBLISHED_PKG))
    from topic_groups import (TOPIC_GROUPS, RESIDUAL, SELECTED, THRESHOLD,
                              TOPIC_COLORS, COMMUNITIES as PUB_COMMUNITIES,
                              COMMUNITY_TAG, months as published_months)
    print(f"loaded the published mapping from {PUBLISHED_PKG}/topic_groups.py")
else:
    GROUPS_FILE = os.path.join(OUTPUT_DIR, "my_topic_groups.json")
    if os.path.exists(GROUPS_FILE):
        with open(GROUPS_FILE, encoding="utf-8") as fh:
            TOPIC_GROUPS = {k: list(v) for k, v in json.load(fh).items()}
        print(f"loaded your mapping from {GROUPS_FILE}")
    else:
        # EDIT THIS. Every topic from step 11 should appear exactly once;
        # the ones you cannot place go in RESIDUAL.
        TOPIC_GROUPS = {
            "Category A": [],
            "Category B": [],
            RESIDUAL: list(range(lda.num_topics)),
        }
        with open(GROUPS_FILE, "w", encoding="utf-8") as fh:
            json.dump(TOPIC_GROUPS, fh, ensure_ascii=False, indent=2)
        print(f"a starter mapping is at {GROUPS_FILE}.")
        print("Edit it — or edit the dict in this cell — then run this cell again.")
    THRESHOLD = 1.25        # step 14 explains this
    SELECTED  = None        # step 14 derives it from your data
    TOPIC_COLORS = None
    PUB_COMMUNITIES = COMMUNITY_TAG = published_months = None

In [ ]:
# Check the mapping before trusting anything built on it.
placed = [t for ids in TOPIC_GROUPS.values() for t in ids]
missing   = sorted(set(range(lda.num_topics)) - set(placed))
duplicate = sorted({t for t in placed if placed.count(t) > 1})

print(f"{len(TOPIC_GROUPS)} categories over {len(placed)} topic slots")
if missing:
    print(f"NOT PLACED: {missing}  <-- these topics are in no category")
if duplicate:
    print(f"IN TWO CATEGORIES: {duplicate}  <-- a tweet would be counted twice")
if not missing and not duplicate:
    print("every topic is in exactly one category")
if RESIDUAL in TOPIC_GROUPS:
    print(f"'{RESIDUAL}' holds {len(TOPIC_GROUPS[RESIDUAL])} topics and is "
          f"excluded from every denominator below")
else:
    print(f"no '{RESIDUAL}' category: shares will be of all placed tweets")

CATEGORY_OF = {t: c for c, ids in TOPIC_GROUPS.items() for t in ids}

rows = []
for c, ids in sorted(TOPIC_GROUPS.items()):
    rows.append({"category": c, "topics": len(ids),
                 "ids": ", ".join(map(str, sorted(ids)))})
mapping_table = pd.DataFrame(rows).set_index("category")
mapping_table

In [ ]:
# The categories in their own words: each one's topics, side by side.
shown = 0
for c, ids in sorted(TOPIC_GROUPS.items()):
    if c == RESIDUAL or not ids:
        continue
    shown += 1
    print(f"\n{c}")
    for t in sorted(ids):
        print(f"   topic {t:2d}: " + ", ".join(TOPIC_WORDS[t][:8]))
if shown == 0:
    print("Every topic is still in the residual category, so there is nothing")
    print("to show. Fill in the mapping in step 12 and run it again — nothing")
    print("below this point will produce a figure until you do.")

---
## 13. Assign every tweet, and group the tweets by community

LDA gives each tweet a *mixture* over topics. The chapter takes the single
strongest topic and discards the rest — a real simplification, since a tweet that
is 0.31 politics and 0.29 religion is counted wholly as politics, and one better
stated than hidden.

The community labels are **taken as given** from the earlier published study,
which built them from who retweets whom. Note what that means: membership is
amplification, not authorship, and a user can belong to more than one community.
If your tweets carry no community field, the corpus is treated as one group
called `All` and every later step still runs.

In [ ]:
def community_members(frame):
    """-> {community name: boolean mask over frame}"""
    if MODE == "chapter" and COMMUNITY_FIELD in frame.columns:
        # the published names and the tag each one carries in `retweeters`
        col = frame[COMMUNITY_FIELD]
        print("using the four published communities and their tags")
        out = {}
        for name in PUB_COMMUNITIES:
            tag = COMMUNITY_TAG[name]
            out[name] = col.map(lambda v, t=tag:
                                t in v if isinstance(v, (list, tuple, set))
                                else str(v) == t)
        return out

    if not COMMUNITY_FIELD or COMMUNITY_FIELD not in frame.columns:
        print("no community field -> the corpus is treated as one group, 'All'")
        return {"All": pd.Series(True, index=frame.index)}

    col = frame[COMMUNITY_FIELD]
    sample = next((v for v in col if isinstance(v, (list, tuple, set, str))), None)

    if isinstance(sample, (list, tuple, set)):
        # a list of tags per tweet: a tweet can be in several communities
        tags = sorted({t for v in col if isinstance(v, (list, tuple, set))
                       for t in v})
        print(f"'{COMMUNITY_FIELD}' holds a list per tweet; {len(tags)} distinct tags")
        if len(tags) > 12:
            print(f"showing the 12 most common of {len(tags)}")
            counts = pd.Series([t for v in col
                                if isinstance(v, (list, tuple, set)) for t in v]
                               ).value_counts()
            tags = list(counts.head(12).index)
        return {t: col.map(lambda v, t=t: isinstance(v, (list, tuple, set))
                           and t in v) for t in tags}

    # one label per tweet
    vals = col.astype(str).value_counts()
    print(f"'{COMMUNITY_FIELD}' holds one label per tweet; {len(vals)} distinct")
    keep = list(vals.head(12).index)
    return {v: col.astype(str) == v for v in keep}

MEMBERS = community_members(df)
COMMUNITIES = list(MEMBERS)
print()
for c, mask in MEMBERS.items():
    print(f"   {c:16} {int(mask.sum()):,} tweets")
overlap = sum(int(m.sum()) for m in MEMBERS.values())
if overlap > len(df):
    print(f"\nthe groups overlap: {overlap:,} memberships over {len(df):,} tweets.")
    print("A tweet retweeted inside two communities is counted in both.")

In [ ]:
# Guard: every word id must be inside the model's topic matrix. If the bags of
# words were built against a different vocabulary, the line below says so here
# instead of letting numpy raise "index N is out of bounds for axis 1".
vocab_n = lda.get_topics().shape[1]
max_id = max((w for b in df["bow"] for w, _ in b), default=-1)
if max_id >= vocab_n:
    raise SystemExit(
        f"The bags of words use word ids up to {max_id:,}, but this model only "
        f"knows {vocab_n:,} words.\nThe vocabulary in step 9 is not the one the "
        f"model was trained on. In \"chapter\" mode step 9 must take the "
        f"vocabulary from the model itself; check that it did.")
print(f"word ids go up to {max_id:,}, model vocabulary is {vocab_n:,} — consistent")


def top_topic_per_tweet():
    """The single strongest topic for every tweet, in corpus order. -1 = none."""
    bows = df["bow"].tolist()
    out, t0, step = [], time.time(), 50000
    for lo in range(0, len(bows), step):
        chunk = bows[lo:lo + step]
        for doc in lda.get_document_topics(chunk):
            out.append(max(doc, key=lambda x: x[1])[0] if doc else -1)
        print(f"   {min(lo+step, len(bows)):,}/{len(bows):,}  "
              f"{time.time()-t0:.0f}s")
    return out

df["topic"] = cached(f"step13_topics_{MODE}_{len(df)}_{len(dictionary)}",
                     top_topic_per_tweet, "topic assignment")
df["category"] = df["topic"].map(lambda t: CATEGORY_OF.get(t))

assigned = (df["topic"] >= 0).sum()
print(f"\n{assigned:,} of {len(df):,} tweets got a topic "
      f"({len(df)-assigned:,} were empty after filtering)")
if assigned == 0:
    print("\nNOT ONE tweet could be assigned. Nothing below will mean anything.")
    print("The cause is upstream, in step 9: the vocabulary is empty or nearly")
    print("so. Re-read what step 9 printed.")
elif df["category"].notna().sum() == 0:
    print("\nTweets have topics but no category: the step 12 mapping is empty.")
print("\ntweets per category:")
print(df["category"].value_counts(dropna=False).to_string())

In [ ]:
# Monthly share of each category, per community.
# The residual category is excluded from the denominator.
def month_keys(frame):
    if "when" not in frame.columns:
        return None
    if MODE == "chapter":
        # The chapter's window is fixed at 42 months. Deriving months from the
        # data would stretch it back to 2008, and the step-14 threshold — a SUM
        # of monthly shares — would then be met by far more categories.
        ms = published_months()
        print(f"the chapter's observation window: {len(ms)} months, "
              f"{ms[0]} .. {ms[-1]}")
        return ms
    ms = sorted({f"{d.year}_{d.month}" for d in frame["when"]},
                key=lambda k: tuple(int(x) for x in k.split("_")))
    return ms

MONTHS = month_keys(df)
if MONTHS is None:
    print("no timestamps -> no monthly figures. Steps 14-17 will be skipped.")
else:
    if MODE != "chapter":
        print(f"{len(MONTHS)} months, {MONTHS[0]} .. {MONTHS[-1]}")
    df["month"] = [f"{d.year}_{d.month}" for d in df["when"]]
    inside = df["month"].isin(MONTHS).sum()
    if inside < len(df):
        print(f"{len(df)-inside:,} of {len(df):,} tweets fall outside the "
              f"window and take no part in the figures")

def shares_for(mask):
    sub = df[mask & df["category"].notna()]
    counted = sub[sub["category"] != RESIDUAL]
    tab = (counted.groupby(["month", "category"]).size()
           .unstack(fill_value=0).reindex(MONTHS, fill_value=0))
    cats = [c for c in TOPIC_GROUPS if c != RESIDUAL]
    tab = tab.reindex(columns=cats, fill_value=0)
    total = tab.sum(axis=1)
    # A month with no classified tweet divides by zero. Masking the zero with
    # where() keeps the frame numeric; replace(0, pd.NA) would turn it to
    # object dtype and matplotlib then refuses it as a marker size.
    out = tab.div(total.where(total != 0), axis=0).fillna(0.0)
    return out.astype(float)

if MONTHS is not None:
    SHARES = {c: shares_for(m) for c, m in MEMBERS.items()}
    for c, frame in SHARES.items():
        path = os.path.join(OUTPUT_DIR, f"Topics_Discussed_by_{c}.csv")
        frame.to_csv(path, index_label="Year-month")
    print(f"wrote {len(SHARES)} CSVs to {OUTPUT_DIR}")
    display(SHARES[COMMUNITIES[0]].head())

---
## 14. Which categories are worth plotting

Twenty-odd categories on one figure is unreadable, so the chapter keeps only the
ones that are consistently present. The rule: **sum a category's monthly shares;
keep it if the sum clears 1.25 in every community.** Over 42 months that sum is a
mean monthly share of about three per cent.

The threshold is a presentation choice, not a test, and it is stated here so a
reader can change it and see what moves. The table below shows every category's
sum in every community, so you can see which ones sit near the line.

In [ ]:
if MONTHS is None:
    print("skipped: no timestamps")
else:
    sums = pd.DataFrame({c: SHARES[c].sum() for c in COMMUNITIES})
    retention = sums.round(3).copy()
    retention["clears_in"] = (sums >= THRESHOLD).sum(axis=1)
    retention["all"] = retention["clears_in"] == len(COMMUNITIES)
    retention = retention.sort_values(["all", "clears_in"], ascending=False)

    passes_all = [c for c in sums.index if (sums.loc[c] >= THRESHOLD).all()]
    print(f"threshold {THRESHOLD} (about a {THRESHOLD/len(MONTHS)*100:.1f}% "
          f"mean monthly share over {len(MONTHS)} months)")
    print(f"clears it in all {len(COMMUNITIES)} communities: {len(passes_all)}")
    for c in passes_all:
        print(f"   {c}")

    partial = [c for c in sums.index
               if 0 < (sums.loc[c] >= THRESHOLD).sum() < len(COMMUNITIES)]
    if partial:
        print("\nclears it in some but not all:")
        for c in partial:
            where = [x for x in COMMUNITIES if sums.loc[c, x] >= THRESHOLD]
            print(f"   {c} -> {', '.join(where)}")

    if SELECTED is None:
        SELECTED = passes_all
        print(f"\nplotting the {len(SELECTED)} that clear it everywhere")
    else:
        print(f"\nthe chapter plots {len(SELECTED)}")
        for c in SELECTED:
            if c not in passes_all:
                where = [x for x in COMMUNITIES if sums.loc[c, x] >= THRESHOLD]
                print(f"   plotted but not in all: {c} -> {', '.join(where) or 'none'}")
        for c in passes_all:
            if c not in SELECTED:
                print(f"   clears it but not plotted: {c}")
    retention

---
## 15. Figure 15.3 — how much each community talks about what

One bar group per category, one bar per community, height is the mean monthly
share. Where the bars in a group are of unequal height, the communities give the
subject unequal attention.

In [ ]:
if MONTHS is None:
    print("skipped: no timestamps, so there are no monthly shares to average")
elif not SELECTED:
    print("skipped: not one category cleared the threshold, so there is nothing")
    print("to plot. Either the grouping in step 12 is still empty, or the")
    print("vocabulary collapsed in step 9 — check what those cells printed.")
elif not HAVE["matplotlib"]:
    print("matplotlib is missing; here are the numbers instead")
    print(pd.DataFrame({c: SHARES[c][SELECTED].mean()*100
                        for c in COMMUNITIES}).round(2).to_string())
else:
    import matplotlib.pyplot as plt

    means = pd.DataFrame({c: SHARES[c][SELECTED].mean() * 100
                          for c in COMMUNITIES})
    order = means.sum(axis=1).sort_values(ascending=False).index
    means = means.loc[order]

    ax = means.plot.bar(figsize=(13, 5), width=.8, rot=0)
    ax.set_ylabel("Mean monthly share (%)")
    ax.set_xticklabels(range(1, len(order) + 1))
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "figure_15_3.png")
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()

    print("the numbered categories, in the order plotted:")
    for i, cat in enumerate(order, 1):
        print(f"   {i} - {cat}")
    print(f"\n{means.round(2).to_string()}")
    print(f"\nsaved to {out}")

---
## 16. Figure 15.4 — the same shares month by month

One panel per community, one row per category, one bubble per month, bubble size
is that month's share. A row that swells in the same month across all four panels
is a shared shock; a row that swells in one panel only is a divergence.

In [ ]:
if MONTHS is None:
    print("skipped: no timestamps")
elif not SELECTED:
    print("skipped: not one category cleared the threshold — see step 14")
elif not HAVE["matplotlib"]:
    print("matplotlib is missing; the monthly numbers are in the CSVs from step 13")
else:
    import matplotlib.pyplot as plt

    if TOPIC_COLORS:
        colors = {c: TOPIC_COLORS[c] for c in SELECTED}
    else:
        cmap = plt.get_cmap("tab20")
        colors = {c: cmap(i % 20) for i, c in enumerate(SELECTED)}

    labels = [f"{m.split('_')[0]} Q{(int(m.split('_')[1])-1)//3+1}"
              if int(m.split("_")[1]) % 3 == 2 else "" for m in MONTHS]

    # Marker area tracks the share. The chapter's scale of 4000 suits shares of
    # a few per cent; with few categories the shares are larger and would
    # swallow the panel, so outside "chapter" mode it is set from the data.
    peak = max(float(SHARES[c].loc[MONTHS, SELECTED].to_numpy().max())
               for c in COMMUNITIES)
    BUBBLE = 4000 if MODE == "chapter" else min(4000, 0.09 / max(peak, 1e-9) * 4000)
    if MODE != "chapter":
        print(f"biggest monthly share is {peak*100:.1f}%; "
              f"marker scale set to {BUBBLE:.0f}")

    n = len(COMMUNITIES)
    rows = 2 if n > 1 else 1
    cols = (n + rows - 1) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 4*rows),
                             sharex=True, sharey=True, squeeze=False)
    flat = axes.ravel()
    for ax, community in zip(flat, COMMUNITIES):
        for r, cat in enumerate(SELECTED):
            ax.scatter(range(len(MONTHS)), [len(SELECTED) - r] * len(MONTHS),
                       s=SHARES[community].loc[MONTHS, cat] * BUBBLE,
                       color=colors[cat], alpha=.85,
                       edgecolors="white", linewidths=.5)
        ax.set_title(community, fontsize=11)
        ax.set_yticks([]); ax.set_xticks(range(len(MONTHS)))
        ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.spines[["top", "right", "left"]].set_visible(False)
    for ax in flat[n:]:
        ax.set_visible(False)
    fig.legend(handles=[plt.Line2D([], [], marker="o", linestyle="",
                                   color=colors[c], label=c) for c in SELECTED],
               loc="lower center", ncol=3, frameon=False, fontsize=9)
    fig.suptitle("Monthly share of each retained category; marker size is the share",
                 fontsize=11)
    fig.tight_layout(rect=(0, .15, 1, .97))
    out = os.path.join(OUTPUT_DIR, "figure_15_4.png")
    plt.savefig(out, dpi=200)
    plt.show()
    print(f"saved to {out}")

---
## 17. Open a category up

A category is an aggregate, and an aggregate can hide the thing you actually want
to report. Set `INSPECT` to a category name to see its topics separately, month by
month. In the chapter this is what shows that the March 2020 spike in *Suffering*
is one of its two topics and not the other.

In [ ]:
INSPECT = "Suffering" if "Suffering" in TOPIC_GROUPS else (SELECTED[0] if SELECTED else None)

if MONTHS is None or not INSPECT:
    print("skipped")
else:
    ids = TOPIC_GROUPS[INSPECT]
    print(f"'{INSPECT}' is built from {len(ids)} topics:")
    for t in sorted(ids):
        print(f"   topic {t:2d}: " + ", ".join(TOPIC_WORDS[t][:10]))

    MIN_MONTH = 100     # months with fewer classified tweets are left blank
    for t in sorted(ids):
        rows, thin = {}, {}
        for c, mask in MEMBERS.items():
            sub = df[mask & df["category"].notna()]
            counted = sub[sub["category"] != RESIDUAL]
            denom = counted.groupby("month").size().reindex(MONTHS, fill_value=0)
            num = (sub[sub["topic"] == t].groupby("month").size()
                   .reindex(MONTHS, fill_value=0))
            # A month with only a few classified tweets produces a share like
            # 50% from two tweets. Mask anything under MIN_MONTH rather than
            # letting it stand as a finding.
            rows[c] = (num / denom.where(denom >= MIN_MONTH) * 100
                       ).fillna(0.0).astype(float)
            thin[c] = int((denom < MIN_MONTH).sum())
        frame = pd.DataFrame(rows).loc[MONTHS]
        peak = frame.mean(axis=1).idxmax()
        print(f"\ntopic {t}: share of topic-assigned tweets (%), "
              f"biggest month {peak}")
        if any(thin.values()):
            print(f"   (months with fewer than {MIN_MONTH} classified tweets "
                  f"are shown as 0: " +
                  ", ".join(f"{c} {n}" for c, n in thin.items() if n) + ")")
        around = [m for m in MONTHS
                  if abs(MONTHS.index(m) - MONTHS.index(peak)) <= 2]
        print(frame.loc[around].round(2).to_string())

---
## 18. Does it match the chapter?

`"chapter"` mode ends by comparing what this notebook just computed against the
CSVs published with the chapter. The two will not be bit-identical: gensim
reseeds the variational step on each inference pass, so repeated runs of the same
saved model on the same tweets differ by a few hundredths of a percentage point.
Anything under about 0.05 pp means the recomputation agrees.

In [ ]:
pub_dir = os.path.join(PUBLISHED_PKG, "data")
if MODE != "chapter":
    print('skipped: MODE is "own", so there is nothing published to compare with.')
elif MONTHS is None or not os.path.isdir(pub_dir):
    print(f"skipped: {pub_dir} not found")
else:
    worst, worst_where, compared = 0.0, "", 0
    for community in COMMUNITIES:
        path = os.path.join(pub_dir, f"Topics_Discussed_by_{community}.csv")
        if not os.path.exists(path):
            print(f"{community:14} no published CSV at {path}"); continue
        pub = pd.read_csv(path).set_index("Year-month")
        cols = [c for c in SELECTED if c in pub.columns]
        mine = SHARES[community][cols].mean() * 100
        theirs = pub[cols].mean() * 100
        diff = (mine - theirs).abs()
        compared += 1
        if diff.max() > worst:
            worst, worst_where = diff.max(), f"{community}/{diff.idxmax()}"
        print(f"{community:14} largest difference {diff.max():.3f} pp "
              f"({diff.idxmax()})")

    if compared == 0:
        print("\nNOTHING WAS COMPARED. Not one published CSV was found, so this")
        print("step establishes nothing at all — it is not a pass. Check that")
        print(f"{pub_dir} holds Topics_Discussed_by_<community>.csv for the")
        print("names printed in step 13.")
    else:
        print(f"\n{compared} of {len(COMMUNITIES)} communities compared")
        print(f"largest difference anywhere: {worst:.3f} pp   [{worst_where}]")
        if worst < 0.05:
            print("under about 0.05 pp: this run agrees with the chapter.")
        else:
            print("this is too large to be gensim's own variation. The")
            print("preprocessing in steps 4 and 7 is not the preprocessing")
            print("behind the chapter — check what those steps reported.")

---
## What this notebook does and does not establish

**It does** take you from a folder of tweets to the two published figures with
every intermediate number computed in front of you, and it makes the two human
decisions — which topics go in which category, and which categories are worth
plotting — visible and editable rather than buried.

**It does not** make the chapter's model reproducible. That model was trained
without a seed; `"chapter"` mode loads it. Your own run in `"own"` mode *is*
reproducible.

**It does not** validate the community labels. They come from a previously
published study, they describe who retweets whom rather than who writes what, and
a user can sit in more than one. Any claim about a community's beliefs rests on
that study, not on anything computed here.

**It does not** justify the single-topic assignment. Re-running step 13 with a
confidence floor, and seeing how far the figures move, is the obvious robustness
check. It is not done here.

**It does not** turn the 1.25 retention threshold into a test. It is a
presentation choice; step 14 prints every category's sums so you can move the
line and see what changes.